# 智能家居 × Jev：从官方案例到我们自己的 3D 演练场

本笔记本分四部分：**① 官方案例讲解 → ② 实现原理与逻辑 → ③ 我们的实验 → ④ 启动 Web 端体验**（末尾把整个 3D 应用内嵌进笔记本）。

运行前置：本目录下已启动 `serve_smart_home.py`（第 4 节的启动单元格也可以一键拉起）。

## ① 官方案例讲解

TypeSafe 官方的智能家居演示（教学视频）做了四件事：

1. **单次调用多重评估**：用户说 "get the coffee boiling"，系统只调用 TypeSafe API **一次**，
   却在同一发请求里塞进一大批问题：意图是什么？是不是复合指令？作用范围？哪类设备？哪个具体设备？
2. **choice 原语定意图**：`choice` 返回每个选项的概率分布——
   smart_home_command 99% / info_request / smart_home_query，取 argmax 后由代码分支。
3. **投机提示（Speculative Prompting）**：明明只涉及咖啡机，也"顺带"问了门锁——返回
   "79% unlock / 21% lock"这种不确定分布。视频里的原话是：反正快且便宜，问都问了，
   真用到门锁的指令来时就不用再跑一趟。**代码端剪枝**掉不相关的答案即可。
4. **该出手的才出手**：复合指令（"关厨房灯然后锁书房门"，复合概率 98%）先用 178ms 的
   TypeSafe 初筛，再交 Claude Haiku 拆成两条原子指令，并行回 TypeSafe 评估；
   常识问题（"1989 世界大赛冠军"）则路由给大模型回答——**快而确定的走 Jev，开放式的走 LLM**。

## ② 实现原理与逻辑

### 2.1 一次请求长什么样

`POST /v1/systemone`，请求 = `state`（当前世界状态）+ `questions`（问题包）；
回复 = 每个问题的答案，带完整概率分布：

In [1]:
import json, urllib.request

payload = {
  "state": {"utterance": "把咖啡烧上", "home": {"rooms": ["living_room", "kitchen"]}},
  "model": "jev-latest",
  "questions": {
    "intent": {"type": "choice", "criteria": {
        "smart_home_command": "控制设备", "information_request": "与家居无关的常识"}},
    "category": {"type": "choice", "criteria": {
        "lighting": "灯", "appliance": "家电", "not_device_specific": "不涉及"}},
    "appliance_action": {"type": "choice", "criteria": {
        "turn_on": "打开", "turn_off": "关闭", "leave_unchanged": "保持"}}
  }
}
req = urllib.request.Request("http://127.0.0.1:8810/api/jev",
    data=json.dumps(payload).encode(), headers={"Content-Type": "application/json"})
wire = json.loads(urllib.request.urlopen(req, timeout=20).read())
print(f"上游耗时 {wire['upstream_ms']}ms")
for name, a in wire["body"]["answers"].items():
    dist = a.get("probabilities") or {"noul": a.get("noul")}
    print(f"  {name:18s} -> {a.get('choice', '—'):22s} {json.dumps(dist, ensure_ascii=False)}")

上游耗时 201ms
  intent             -> smart_home_command     {"smart_home_command": 0.97, "information_request": 0.02, "smart_home_query": 0.01}
  is_compound        -> —                      {"noul": 0.074}
  scope              -> specific_device        {"whole_house": 0.05, "single_room": 0.05, "specific_device": 0.9}
  target_room        -> not_room_specific      {"living_room": 0.02, "kitchen": 0.02, "office": 0.02, "bedroom": 0.02, "entrance": 0.02, "not_room_specific": 0.9}
  category           -> appliance              {"lighting": 0.02, "appliance": 0.902, "lock": 0.02, "fan": 0.02, "speaker": 0.02, "not_device_specific": 0.02}
  light_action       -> turn_on                {"turn_on": 0.942, "turn_off": 0.029, "leave_unchanged": 0.029}
  fan_action         -> turn_on                {"turn_on": 0.907, "turn_off": 0.047, "leave_unchanged": 0.047}
  speaker_action     -> turn_on                {"turn_on": 0.906, "turn_off": 0.047, "leave_unchanged": 0.047}
  appliance_action   -> tu

### 2.2 三种原语就是全部词汇

| 原语 | 回答什么 | 返回 |
|---|---|---|
| `choice` | 多选一（意图/设备类别/动作） | argmax + 全分布 + 置信 |
| `noul` | 是/否（复合吗？A 必须先于 B 吗？） | 一个 0~1 概率 |
| `score` | 有序评分（优先级/紧急度） | 期望分 + 各档概率 |

没有文本生成、没有对话——所以快、便宜、输出天然结构化。**开放式的活再转包给大模型**。

### 2.3 我们页面里的完整管线

```
指令 ──► ① Jev 投机调用（10 问捆绑：意图/复合/范围/房间/类别 + 五类设备动作）
          ├─ intent=信息请求 ──► step-3.5-flash 直答（Jev 只当 141ms 的路由守卫）
          ├─ intent=状态查询 ──► 投机答案定位 + 本地 3D 状态直读（零额外调用）
          └─ intent=控制指令
               ├─ is_compound ≥ 50% ──► LLM 拆原子指令
               │      └─ ② Jev 编排调用：每对子指令 noul「A 必须先于 B 吗」
               │           + 每条 score 优先级（安全最先）
               │           任一依赖 ≥60% → 串行执行栈（拓扑序逐个派发）
               │           全部 <60%    → 并行执行
               └─ 单一指令 ──► 直接取投机答案里相关类别的动作派发（剪枝其余）
```

**串行/并行的意义**：「先开客厅灯再关掉」两步作用于同一设备，顺序决定最终状态 → 必须串行；
「关厨房灯 + 锁书房门」互不相关 → 并行省时间。依赖判定这件事本身也交给 Jev 的原语完成。

## ③ 我们的实验：智能家居 3D 演练场

参考官方案例 UI 复刻的 Three.js 三维户型（客厅/厨房/书房/卧室/玄关，设备可点选），
把演示视频里的"假数据"全部换成真实调用：

- **语音直接输入**：阶跃星辰流式 ASR（`stepaudio-2.5-asr`，16k 裸 PCM，首字 ~0.52s），
  中英文自动检测；打字输入保留。
- **串行/并行任务编排**：同灯先开后关判 93% 依赖 → 串行栈执行，最终灯为关（顺序保持）；
  异类指令并行 + 按优先级排序。编排分析仅 133–141ms。
- **传统 LLM 对照**：一键切换 step-5-preview 一次直答做同样的事——JSON 遵从完美，
  但延迟 3–20×、每单多几百输出 token（页面右下角实时显示成本：Jev 输入 $0.042/M、输出免费）。
- **无 Key 降级**：不设 `TYPESAFE_API_KEY` 自动切本地模拟引擎，功能照常。

实测明细见 [README.md](README.md)。

## ④ 启动 Web 端 + 内嵌体验

可以内嵌：我们的本地服务不发 `X-Frame-Options`，Jupyter 里直接用 `<iframe>` 即可。
注意要加 `allow="microphone"`，否则框里麦克风会被浏览器禁掉。

In [2]:
import subprocess, sys, time, urllib.request, os
from pathlib import Path

PORT = 8810
BASE = f"http://127.0.0.1:{PORT}"

def server_up():
    try:
        return urllib.request.urlopen(f"{BASE}/api/health", timeout=2).status == 200
    except Exception:
        return False

if not server_up():
    env = dict(os.environ)
    subprocess.Popen([sys.executable, "serve_smart_home.py", "--port", str(PORT),
                      "--host", "127.0.0.1", "--no-open"],
                     cwd=str(Path.cwd()), env=env,
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(30):
        if server_up():
            break
        time.sleep(0.5)

import json
health = json.loads(urllib.request.urlopen(f"{BASE}/api/health", timeout=2).read())
print("服务状态:", "运行中" if server_up() else "启动失败")
print("  真实 Jev :", "已配置" if health["jev"] else "未配置（页面用本地模拟引擎）")
print("  阶跃 ASR :", health["asr_model"] if health["asr"] else "未配置")

服务状态: 运行中
  真实 Jev : 已配置
  阶跃 ASR : stepaudio-2.5-asr


In [3]:
from IPython.display import HTML
HTML(f'''
<div style="border:1px solid #334155;border-radius:12px;overflow:hidden">
  <iframe src="{BASE}/" style="width:100%;height:920px;border:0"
          allow="microphone"></iframe>
</div>
<p style="font-size:12px;color:#64748b">3D 场景可拖拽旋转；麦克风需在
<a href="{BASE}/" target="_blank">独立标签页</a>打开时体验更稳（部分浏览器限制 iframe 内的录音权限）。</p>
''')

## 尾注

- 代码与实验报告：[Bald0Wang/jev-playground → smart-home/](https://github.com/Bald0Wang/jev-playground)
- Jev 定价：输入 $0.042/M tokens、输出免费；阶跃定价见官方 `docs/zh/guides/pricing/details`
- 环境搭建：`TYPESAFE_API_KEY`（console.typesafe.ai）、`STEPFUN_API_KEY`（platform.stepfun.com）